In [ ]:
!pip install anthropic playwright
!playwright install chromium

In [2]:
import os
import subprocess
import json
import getpass
from anthropic import Anthropic

# 1. Initialize API Key as a string (using secure input so it's not hardcoded)
api_key = getpass.getpass("Enter your Anthropic API Key: ")
client = Anthropic(api_key=api_key)

# 2. Define the absolute path for our workspace file
TARGET_FILE = os.path.abspath("test_app.py")

# 3. Create the initial BROKEN test file programmatically
broken_code = """import sys
from playwright.sync_api import sync_playwright

def test_search_feature():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

        # Navigate to a real, stable public sandbox
        page.goto("https://example.com")
        print("STEP 1: Successfully navigated to example.com")

        # INTENTIONAL BUG: "#broken-search-input-id" does not exist on example.com.
        # FIX REQUIRED: To make this pass smoothly, change this selector to "h1" (which exists) 
        # and change page.click to page.inner_text or page.is_visible.
        print("STEP 2: Attempting to click the search input...")
        page.click("#broken-search-input-id", timeout=3000) 

        print("STEP 3: Action executed successfully.")
        browser.close()

if __name__ == "__main__":
    try:
        test_search_feature()
        print("RESULT: All tests passed successfully!")
        sys.exit(0)
    except Exception as e:
        print(f"RESULT: Test Suite Failed! Error Details: {str(e)}")
        sys.exit(1)
"""

with open(TARGET_FILE, "w") as f:
    f.write(broken_code)

print(f"Initialization complete! Created broken test file at: {TARGET_FILE}")

Initialization complete! Created broken test file at: /Users/amritansh/Documents/EY_AI_Test/D4_EY/test_app.py


In [4]:
def read_file():
    """Reads the current contents of the test script."""
    if os.path.exists(TARGET_FILE):
        with open(TARGET_FILE, "r") as f:
            return f.read()
    return "Error: File not found."

def write_file(content):
    """Rewrites the test script with updated content."""
    with open(TARGET_FILE, "w") as f:
        f.write(content)
    return "File successfully updated and saved."

def run_test_script():
    """Runs the test script and returns the exact stdout and stderr output."""
    result = subprocess.run(
        ["python3", TARGET_FILE],
        capture_output=True,
        text=True
    )
    return f"Exit Code: {result.returncode}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"

# Define tool schemas for the Anthropic Claude API
tools_schema = [
    {
        "name": "read_file",
        "description": "Read the contents of the local 'test_app.py' test script file.",
        "input_schema": {"type": "object", "properties": {}}
    },
    {
        "name": "write_file",
        "description": "Rewrite the contents of the local 'test_app.py' test script file to fix bugs.",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "The complete, revised Python code."}
            },
            "required": ["content"]
        }
    },
    {
        "name": "run_test_script",
        "description": "Run 'test_app.py' inside the current environment to verify if tests pass or fail.",
        "input_schema": {"type": "object", "properties": {}}
    }
]

In [5]:
# System prompt to give Claude its QA Engineer persona and goals
SYSTEM_PROMPT = """You are an autonomous AI QA Engineer. Your goal is to inspect the test script 'test_app.py', run it using your tools, detect any hidden element identification bugs or crashes, correct the code yourself, and verify that it completes with an exit code of 0. Do not stop until the test passes.

CRITICAL: When fixing 'test_app.py' to run cleanly against 'https://example.com', remember that example.com contains no input text elements. You can look for the 'h1' tag or read text contents instead to ensure a successful pass."""

# Initialize conversation history
messages = [
    {"role": "user", "content": "Start the QA optimization loop. Run the test, find the error, fix it, and confirm success."}
]

running = True
step_counter = 1

while running:
    print(f"\n--- 🤖 [STEP {step_counter}]: Claude is evaluating the environment... ---")
    
    # 1. Request response from Claude (Using correct, valid model string)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=messages,
        tools=tools_schema
    )
    
    # Track the assistant's turn response content
    assistant_content = response.content
    
    # Print any thoughts or statements Claude made
    for content_block in assistant_content:
        if content_block.type == "text":
            print(f"\n[Claude]: {content_block.text}")
            
    # Filter out if there are any tool calls
    tool_use_blocks = [b for b in assistant_content if b.type == "tool_use"]
    
    # Append Claude's response (including the tool_use blocks) directly to history first
    messages.append({"role": "assistant", "content": assistant_content})
    
    if not tool_use_blocks:
        print("\n🎉 [Agent Complete]: Claude has completed its tasks without any further tool actions.")
        running = False
        break
        
    # 2. Build the tool results block for the IMMEDIATE NEXT user message turn
    tool_responses = []
    
    for tool_call in tool_use_blocks:
        tool_name = tool_call.name
        tool_args = tool_call.input
        tool_id = tool_call.id
        
        print(f"\n👉 [Executing Tool]: '{tool_name}' automatically...")
        if tool_args:
            print(f"Arguments: {json.dumps(tool_args, indent=2)}")
            
        # Execute the designated tool automatically without stopping
        if tool_name == "read_file":
            result_output = read_file()
        elif tool_name == "write_file":
            result_output = write_file(tool_args["content"])
        elif tool_name == "run_test_script":
            result_output = run_test_script()
        else:
            result_output = f"Error: Unknown tool '{tool_name}'."
            
        print(f"💻 [Tool Output Result]:\n{result_output}\n")
        
        # Structure the tool output format correctly matching its ID
        tool_responses.append({
            "type": "tool_result",
            "tool_use_id": tool_id,
            "content": result_output
        })
        
    # Append the completed tool executions as a single 'user' response turn right after
    messages.append({
        "role": "user",
        "content": tool_responses
    })
        
    step_counter += 1


--- 🤖 [STEP 1]: Claude is evaluating the environment... ---

[Claude]: Sure! Let me start by reading the test file and running it simultaneously to gather all the information I need at once.

👉 [Executing Tool]: 'read_file' automatically...
💻 [Tool Output Result]:
import sys
from playwright.sync_api import sync_playwright

def test_search_feature():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

        # Navigate to a real, stable public sandbox
        page.goto("https://example.com")
        print("STEP 1: Successfully navigated to example.com")

        # INTENTIONAL BUG: "#broken-search-input-id" does not exist on example.com.
        # FIX REQUIRED: To make this pass smoothly, change this selector to "h1" (which exists) 
        # and change page.click to page.inner_text or page.is_visible.
        print("STEP 2: Attempting to click the search input...")
        page.click("#broken-search-input-id", timeout=